
# 11 — Data mapping and normalization (Cars 4 You)

**Scope:** Perform Mapping and normalization (no modeling).  
Outputs a clean dataset and a JSON processing report for traceability.

ONLY Rulebased Cleaning (no ML), Missing-Handling, Encoding-Preparation.  
No Fit, no Scalers, no Target.
We want to avoid any data leakage.

That is the reason, why we don't have to split the data into train/test here.

# Table of Contents


<a class="anchor" id="top"></a>

** **

1. [Importing Libraries](##1.-Importing-Libraries) <br>
    
2. [Data Access & Loading](#2.-Data-Access-&-Loading) <br>
    
3. [Type Casting](#3.-Type-Casting) <br>

4. [Duplicate Removal](#3.1-Duplicate-Removal) <br>
    
5. [Category Normalization](#3.2-Category-Normalization) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

## 1. Import Libraries and functions

In [37]:
import os, re, math, warnings, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

RANDOM_STATE = 42  # for reproducibility of any sampling

In [ ]:
def to_int_series(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

def to_float_series(s):
    return pd.to_numeric(s, errors="coerce").astype(float)


def conversion_dtypes(obj):
    # --- Handle Series input (this is your y column) ---
    if isinstance(obj, pd.Series):
        s = obj.copy()

        # Clean string NaN-like values
        s = (
            s.astype(object)
             .astype(str)
             .str.strip()
             .replace({
                 "nan": np.nan, "NaN": np.nan, "None": np.nan, "NA": np.nan,
                 "<NA>": np.nan, "": np.nan, "null": np.nan, "NULL": np.nan
             })
        )

        # Convert to float (price)
        s = pd.to_numeric(s, errors="coerce").astype(float)
        return s

    # --- Handle DataFrame input ---
    df = obj.copy()
    nan_report = {}

    num_cols = ["price", "mileage", "engineSize", "mpg", "tax", "year", "previousOwners"]

    # --- NUMERIC PROCESSING ---
    for col in num_cols:
        if col in df.columns:

            before = df[col].isna().sum()

            df[col] = (
                df[col]
                .astype(object)
                .astype(str)
                .str.strip()
                .replace({
                    "nan": np.nan, "NaN": np.nan, "None": np.nan,
                    "NA": np.nan, "<NA>": np.nan, "": np.nan,
                    "null": np.nan, "NULL": np.nan,
                })
            )

            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

            after = df[col].isna().sum()
            if after > before:
                nan_report[col] = after - before


    # --- OBJECT PROCESSING ---
    for col in df.select_dtypes(include=["object", "string"]).columns:
        before = df[col].isna().sum()

        df[col] = (
            df[col]
            .astype(object)
            .astype(str)
            .str.strip()
            .replace({
                "nan": np.nan, "NaN": np.nan, "None": np.nan,
                "NA": np.nan, "<NA>": np.nan, "": np.nan,
                "null": np.nan, "NULL": np.nan,
            })
        )

        after = df[col].isna().sum()
        if after > before:
            nan_report[col] = after - before


    if nan_report:
        print("Columns where casting/cleaning created new NaNs:")
        for col, n in nan_report.items():
            print(f"  - {col}: +{n} NaNs")
    else:
        print("No new NaNs created by casting/cleaning.")
    print("df Dtypes after conversion:")
    print(df.dtypes)
    return df

In [ ]:
 def apply_regex_first(val: str) -> str:
        """
        Apply regex-based mapping rules to a normalized model value.

        Args:
            val (str): A normalized model string (output of `norm_model`), e.g., 'a180'.

        Returns:
            str: Transformed canonical string if regex matches; else original value.
        """
        for rule in regex_rules:
            pat = rule["pattern"]
            repl = rule["replace"]
            m = re.fullmatch(pat, val)
            if m:
                return m.expand(repl)
        return val

In [ ]:
def normalize_transmission(value: str):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    value = re.sub(r"[.,_]", " ", value)
    value = " ".join(value.split())
    return value

def apply_transmission_mapping(df: pd.DataFrame,
                               mapping_path: str = "../mapping/transmission_mapping.json",
                               col: str = "transmission") -> pd.DataFrame:
    # load + normalize mapping keys
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_map = json.load(f)

    trans_canon = {normalize_transmission(k): v for k, v in raw_map.items()}


    # debug before
    # uniq_before = df[col].nunique(dropna=True)
    # print(f"[DEBUG] {col}: unique values BEFORE mapping: {uniq_before}")


    # normalize column
    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_transmission)

    # map
    df[col] = df[norm_col].map(trans_canon)
    
    df[col] = df[col].fillna("Unknown")

    # debug after
    # uniq_after = df[col].nunique(dropna=True)
    # print(f"[DEBUG] {col}: unique values AFTER mapping: {uniq_after}")
    # print(f"[DEBUG] {col}: sample AFTER:", df[col].dropna().unique()[:15])

    # check unmapped
    unmapped = (
        df.loc[df[col].isna(), norm_col]
          .dropna()
          .unique()
    )
    if len(unmapped) > 0:
        print(f"[WARN] Unmapped {col} values:")
        for v in unmapped:
            print("  -", repr(v))

    # drop helper
    df.drop(columns=[norm_col], inplace=True)

    return df


In [ ]:
def normalize_fuel(value: str):
    """Normalize raw fuelType text before mapping."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    value = re.sub(r"[.,-_]", " ", value)  # unify separators
    value = " ".join(value.split())        # collapse spaces
    return value

def apply_fuel_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/fueltype_mapping.json",
    col: str = "fuelType"
) -> pd.DataFrame:
    # load + normalize mapping keys
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_fuel_map = json.load(f)

    FUEL_CANON = {normalize_fuel(k): v for k, v in raw_fuel_map.items()}

    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_fuel)

    # map
    df[col] = df[norm_col].map(FUEL_CANON)
    
    df[col] = df[col].fillna("Unknown")


    # report unmapped
    unmapped = (
        df.loc[df[col].isna(), norm_col]
          .dropna()
          .unique()
    )
    if len(unmapped) > 0:
        print(f"[WARN] Unmapped {col} values (add to JSON):")
        for v in unmapped:
            print("  -", repr(v))

    # drop helper
    df.drop(columns=[norm_col], inplace=True)

    return df

In [ ]:
def norm_model(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().lower()
    s = re.sub(r"[.,\-_ ]+", "", s)
    return s

def apply_model_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/modelname_mapping.json",
    col: str = "model"
) -> pd.DataFrame:
    # load JSON
    with open(mapping_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    raw_aliases = cfg["aliases"]
    regex_rules = cfg.get("regex_rules", [])

    # normalize alias keys
    ALIASES = {norm_model(k): v for k, v in raw_aliases.items()}

    if col not in df.columns:
        print(f"[INFO] column '{col}' not in df, skipping.")
        return df


    # normalize incoming values
    df[f"{col}_norm"] = df[col].apply(norm_model)

    # map
    def map_model(norm_val):
        if pd.isna(norm_val):
            return np.nan
        norm_val = apply_regex_first(norm_val)
        return ALIASES.get(norm_val, np.nan)

    df[f"{col}_mapped"] = df[f"{col}_norm"].apply(map_model)

    # overwrite with mapped
    df[col] = df[f"{col}_mapped"]
    
    df[col] = df[col].replace({pd.NA: np.nan})

    # everything that had a value (norm not NA) but no mapping -> set to NA
    mask_unmapped = df[col].isna() & df[f"{col}_norm"].notna()
    df.loc[mask_unmapped, col] = "Unknown"

    # clean up
    df.drop(columns=[f"{col}_norm", f"{col}_mapped"], inplace=True)

    return df

In [ ]:
def normalize_brand(value: str):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    value = re.sub(r"[.,-_]", " ", value)
    value = " ".join(value.split())
    return value

def apply_brand_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/brandname_mapping.json",
    col: str = "Brand"
) -> pd.DataFrame:
    if col not in df.columns:
        print(f"[INFO] Column '{col}' not found.")
        return df

    # load mapping
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_brand_map = json.load(f)

    # normalize mapping keys
    BRAND_CANON = {normalize_brand(k): v for k, v in raw_brand_map.items()}

    was_nan_before = df[col].isna()

    # normalize column
    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_brand)

    # map
    mapped = df[norm_col].map(BRAND_CANON)
    
    df[col] = df[col].fillna("Unknown")

    # keep original where no mapping exists
    clean_col = f"{col}_clean"
    df[clean_col] = np.where(
    mapped.isna(),
    df[col],        # original value
    mapped.astype("object")  # mapped value
    )

    df[clean_col] = df[clean_col].replace({pd.NA: np.nan})

    # detect new NaNs that came from missing mapping
    new_nans_mask = df[clean_col].isna() & (~was_nan_before)
    new_nans_count = new_nans_mask.sum()

    # collect unmapped originals
    unmapped = (
        df.loc[mapped.isna() & (~was_nan_before) & df[norm_col].notna(), norm_col]
          .dropna()
          .unique()
    )

    # replace original
    df[col] = df[clean_col]
    df.drop(columns=[norm_col, clean_col], inplace=True)

    if new_nans_count > 0:
        print(f"[WARN] {new_nans_count} rows became NaN due to missing mapping.")
    if len(unmapped) > 0:
        print("\n[WARN] Unmapped brand values (add to JSON):")
        for v in unmapped:
            print("  -", repr(v))

    return df

In [ ]:
def fill_brand_from_model(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/brand_model_mapping.json"
) -> pd.DataFrame:
    """
    Fills df['Brand'] when:
      - Brand is NaN
      - model has a value
      - model exists in the JSON (brand_model_mapping.json)
    """
    # load JSON: {"A1": "Audi", "Golf": "Volkswagen", ...}
    with open(mapping_path, "r", encoding="utf-8") as f:
        model_to_brand = json.load(f)

    # debug before
    before = df["Brand"].isna().sum()

    # rows where Brand is missing but model is present
    mask = df["Brand"].isna() & df["model"].notna()

    # map model -> Brand
    mapped = df.loc[mask, "model"].map(model_to_brand)

    # write back only where we actually found a brand
    fill_mask = mask.copy()
    fill_mask.loc[mask] = mapped.notna()

    df.loc[fill_mask, "Brand"] = mapped[mapped.notna()]

    return df

In [ ]:
def mapping_and_normalization(df: pd.DataFrame) -> pd.DataFrame:
    """ Applies mapping and normalization to the given DataFrame."""
    # print unique values of each column before mapping
    df = apply_fuel_mapping(df)
    df = apply_transmission_mapping(df)
    df = apply_brand_mapping(df) #lowkey unnecessary if fill_brand_from_model is used after
    df = apply_model_mapping(df)
    df = fill_brand_from_model(df)
    return df

## 2. Data Access & Loading

Loading the data from CSV files into pandas DataFrames. 

In [51]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Loaded shape:", train.shape)
print("Loaded test shape:", test.shape)
display(train.head(3))


Loaded shape: (75973, 14)
Loaded test shape: (32567, 13)


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


In [52]:
# Display the NaN values for each column in a table for train set
nan_summary = train.isna().sum().reset_index()
nan_summary.columns = ['Column', 'NaN Count']
nan_summary = nan_summary[nan_summary['NaN Count'] > 0]
nan_summary = nan_summary.sort_values(by='NaN Count', ascending=False)
nan_summary

,Column,NaN Count
9,mpg,7926
8,tax,7904
12,previousOwners,1550
13,hasDamage,1548
11,paintQuality%,1524
5,transmission,1522
1,Brand,1521
2,model,1517
10,engineSize,1516
7,fuelType,1511


## 3. Process anomalies in numerical columns

We shouldn't round the non integer year values (see notebook 00), so we set them to nan.

In [53]:
train.loc[train['year'] % 1 != 0, 'year'] = np.nan
test.loc[test['year'] % 1 != 0, 'year'] = np.nan

The negative mileage values are likely no sign errors (see notebook 00), so we set them to nan.

In [54]:
train.loc[train['mileage'] < 0, 'mileage'] = np.nan
test.loc[test['mileage'] < 0, 'mileage'] = np.nan

The negative tax values are likely no sign errors (see notebook 00), so we set them to nan.

In [55]:
train.loc[train['tax'] < 0, 'tax'] = np.nan
test.loc[test['tax'] < 0, 'tax'] = np.nan

The negative mpg values are likely no sign errors (see notebook 00), so we set them to nan.

In [56]:
train.loc[train['mpg'] < 0, 'mpg'] = np.nan
test.loc[test['mpg'] < 0, 'mpg'] = np.nan

The negative previousOwners are likely data errors (see notebook 00), so we set them to nan.

In [57]:
# it is no sign error, so the values will be set to na for now
train.loc[train['previousOwners'] < 0, 'previousOwners'] = np.nan
test.loc[test['previousOwners'] < 0, 'previousOwners'] = np.nan

Some MPG values are not realistic, so we filtered the values depending on their fuelType. Unrealistic ones are set to nan.

In [58]:
unusual_mpg_mask = ((train['mpg'] > 150) & (train["fuelType"] == "Electric")) | \
       ((train['mpg'] < 70) & (train["fuelType"] == "Electric")) | \
       ((train['mpg'] > 100) & (train["fuelType"] == "Hybrid")) | \
       ((train['mpg'] < 35) & (train["fuelType"] == "Hybrid")) | \
       ((train['mpg'] > 80) & (train["fuelType"] != "Hybrid") & (train["fuelType"] != "Electric")) | \
       ((train['mpg'] < 8) & (train["fuelType"] != "Hybrid")& (train["fuelType"] != "Electric"))
train.loc[unusual_mpg_mask, 'mpg'] = np.nan

unusual_mpg_mask_test = ((test['mpg'] > 150) & (test["fuelType"] == "Electric")) | \
       ((test['mpg'] < 70) & (test["fuelType"] == "Electric")) | \
       ((test['mpg'] > 100) & (test["fuelType"] == "Hybrid")) | \
       ((test['mpg'] < 35) & (test["fuelType"] == "Hybrid")) | \
       ((test['mpg'] > 80) & (test["fuelType"] != "Hybrid") & (test["fuelType"] != "Electric")) | \
       ((test['mpg'] < 8) & (test["fuelType"] != "Hybrid")& (test["fuelType"] != "Electric"))
test.loc[unusual_mpg_mask_test, 'mpg'] = np.nan

The years after 2020 are data errors (see notebook 00), so we set them to nan.

In [59]:
train.loc[train['year'] > 2020, 'year'] = np.nan
test.loc[test['year'] > 2020, 'year'] = np.nan

We will remove PaintQuality% Feature completely, since we don't have that data for our model (see notebook 00).

In [60]:
train.drop(columns=['paintQuality%'], inplace=True)
test.drop(columns=['paintQuality%'], inplace=True)

The non integer previousOwners likely are data errors (see notebook 00), so we set them to nan.

In [61]:
train.loc[train['previousOwners'] % 1 != 0, 'previousOwners'] = np.nan
test.loc[test['previousOwners'] % 1 != 0, 'previousOwners'] = np.nan

## 3. Type Casting

We convert the data types of the features to appropriate types.
For examples convert the years and previous owner to int (there is no half year or half person)

In [ ]:
train = conversion_dtypes(train)
test = conversion_dtypes(test)

train.head(2)

No new NaNs created by casting/cleaning.


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage
0,69512,VW,Golf,2016,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,4,0.0
1,53000,Toyota,Yaris,2019,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,1,0.0


In [63]:
# verify the transformation
train.dtypes

carID                      int64
Brand             string[python]
model             string[python]
year                       Int64
price                    float64
transmission      string[python]
mileage                  float64
fuelType          string[python]
tax                      float64
mpg                      float64
engineSize               float64
previousOwners             Int64
hasDamage                float64
dtype: object

## 4. Set carId as index for datafames

As we confirmed earlier during data exploration, there are no duplicate values in carId, so it can be used as a unique identifier. We can set it as the DataFrame index to enable more efficient data manipulation.

In [64]:
# Set carID as index for training and test dataframes
train.set_index('carID', inplace=True)
test.set_index('carID', inplace=True)

## 5. Category Normalization and Mapping for the train and test dataset

At almost every feature there is some typos at the values. We create a mapping dictionary to correct them. 
The ones we couldnt map we will set them to NaN so we can impute them later.

In [ ]:
# Firstly we make all string columns to lowercase for consistency
for c in train.select_dtypes(include="string").columns:
    train[c] = train[c].str.lower()
for c in test.select_dtypes(include="string").columns:
    test[c] = test[c].str.lower()

In [ ]:
def mapping_and_normalization(df: pd.DataFrame) -> pd.DataFrame:
    """ Applies mapping and normalization to the given DataFrame."""
    # print unique values of each column before mapping
    df = apply_fuel_mapping(df)
    df = apply_transmission_mapping(df)
    df = apply_brand_mapping(df) #lowkey unnecessary if fill_brand_from_model is used after
    df = apply_model_mapping(df)
    df = fill_brand_from_model(df)
    return df

In [ ]:
train= mapping_and_normalization(train)
test= mapping_and_normalization(test)

# print unique values after mapping for each column
for col in train.columns:
    print(f"\nUnique {col} after mapping:\n", train[col].unique().tolist())

As it was mentioned before, we will set unmapped values to NaN for later imputation.

In [ ]:
train.loc[train['transmission'] == 'Unknown', 'transmission'] = np.nan
test.loc[test['transmission'] == 'Unknown', 'transmission'] = np.nan

train.loc[train['fuelType'] == 'Unknown', 'fuelType'] = np.nan
test.loc[test['fuelType'] == 'Unknown', 'fuelType'] = np.nan    

train.loc[train['Brand'] == 'Unknown', 'Brand'] = np.nan
test.loc[test['Brand'] == 'Unknown', 'Brand'] = np.nan

train.loc[train['model'] == 'Unknown', 'model'] = np.nan
test.loc[test['model'] == 'Unknown', 'model'] = np.nan

## 6. Fix outliers 

We fix outliers because extreme or invalid values can distort statistical summaries and negatively affect model performance. With the outliers we either remove them or we put the the values NA so we impute them later

## Year before 2000

In [ ]:
# Show year values before 2000

print("Year values before 2000:")
print(train.loc[train['year'] < 2000, 'year'])

We only have 15 entries for year before 2000. We delete these rows from the dataset as they are not significant enough to keep and might skew our analysis.

In [ ]:
# Drop the entries with year before 2000
train = train[train['year'] >= 2000]
# test = test[test['year'] >= 2000]

## Engine Size below 0.8L and above 6.2L

We have seen in data exploration that there are some outliers in engine size below 1.0L and above 6.2L.
From our domain knowledge, we know that cars in our dataset should have engine sizes between 1.0L and 6.2L.
We will put these values to na, so we can impute them later.

In [ ]:
# Count affected entries
low_engine_size_count = train[train['engineSize'] < 1.0].shape[0]
high_engine_size_count = train[train['engineSize'] > 6.2].shape[0]
print(f"Number of entries with engine size below 1.0L:  {low_engine_size_count}")
print(f"Number of entries with engine size above 6.2L:  {high_engine_size_count}")

# the values just look wrong, so we will set all unusual values to na
train.loc[((train['engineSize'] > 6.2) | (train['engineSize'] < 0.8)), 'engineSize'] = np.nan
test.loc[((test['engineSize'] > 6.2) | (test['engineSize'] < 0.8)), 'engineSize'] = np.nan

## Car Model Kadjar

In [ ]:
# Print all models named 'kadjar'
print("Entries with model 'Kadjar':")
display(train[train['model'] == 'Kadjar'])

- We only have 3 models named Kadjar.The Model belongs to the Brand Renault, but we don't have Renault in our brand list or any other model from Renault.
- Therefore we don't know if it is really a Kadjar or a typo for another model.
- We will put these values to na, so we can impute them later.

In [ ]:
# Put all Kadjar models to NaN
train.loc[train['model'] == 'Kadjar', 'model'] = np.nan
test.loc[test['model'] == 'Kadjar', 'model'] = np.nan

## 10. Save Processed Data

In [ ]:
train = train.reset_index().rename(columns={'index': 'CarID'})
test = test.reset_index().rename(columns={'index': 'CarID'})

In [ ]:
# Get current date string
date_str = datetime.now().strftime("%Y%m%d_%H%M")

# save the processed dataframe to data/processed_data for train and test
PROCESSED_CSV = os.path.join(data_dir, f"processed_data/{date_str}/11_processed_train_data.csv")
output_dir = os.path.dirname(PROCESSED_CSV)
os.makedirs(output_dir, exist_ok=True)

print("Saving processed file to:", PROCESSED_CSV)
train.to_csv(PROCESSED_CSV, index=False)

# Save the test set
PROCESSED_CSV_TEST = os.path.join(data_dir, f"processed_data/{date_str}/11_processed_test_data.csv")
output_dir_test = os.path.dirname(PROCESSED_CSV_TEST)
os.makedirs(output_dir_test, exist_ok=True)
print("Saving processed file to:", PROCESSED_CSV_TEST)
test.to_csv(PROCESSED_CSV_TEST, index=False)

print("Processed train shape:", train.shape)
print("Processed test shape:", test.shape)

In [ ]:
# Display the NaN values for each column in a table for train set
nan_summary = train.isna().sum().reset_index()
nan_summary.columns = ['Column', 'NaN Count']
nan_summary = nan_summary[nan_summary['NaN Count'] > 0]
nan_summary = nan_summary.sort_values(by='NaN Count', ascending=False)
nan_summary

In [ ]:
# Display the NaN values for each column in a table for test set
nan_summary_test = test.isna().sum().reset_index()
nan_summary_test.columns = ['Column', 'NaN Count']
nan_summary_test = nan_summary_test[nan_summary_test['NaN Count'] > 0]
nan_summary_test = nan_summary_test.sort_values(by='NaN Count', ascending=False)
nan_summary_test
